## Bugs
- 1470 days hard-coded
- At the end ofsql operations, should we close connections?

In [ ]:
import sqlite3
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("star_functions.py"), '..')))
import star_functions as nana
import numpy as np
import matplotlib.pyplot as plt
import lightkurve as lk

In [ ]:
def get_db_connection():
    # return db.connector.connect(
    #     host='localhost',
    #     user='nana',
    #     database='stars_db') this is mysql stuff here
    # conn = db.connect('modes.db.2025-12-28.db', timeout=120.0)
    #aaamodes_copy.db2025-12-28
    conn = sqlite3.connect('/Users/nana/venv/hoggnation/oscillator_catalog/2026-02-22modes.db', timeout=120.0)

    conn.execute("PRAGMA journal_mode=WAL")
    return conn

In [ ]:
#select stars with sqlite3 query
conn = get_db_connection()
cursor = conn.cursor()

cursor.execute("SELECT DISTINCT star_id FROM mode")
stars = cursor.fetchall()


for star in stars:
    
    cursor.execute(f"SELECT frequency, mode_id FROM mode WHERE star_id = '{star[0]}'") 
    results = np.array(cursor.fetchall())
    modes = results[:,0]
    mode_ids = results[:,1]

    ind = np.argsort(modes)
    modes = modes[ind]
    mode_ids = mode_ids[ind]
    parents = np.zeros_like(modes).astype(int)-1
    
    for i, mode in enumerate(modes):
        if parents[i] < 0:
            for j in range(i+1, len(modes)):
                q = np.round(modes[j]/modes[i]) 
                if np.abs(modes[j] -  q*modes[i]) < 1/1470: #this is dif for each star
                    parents[j] = mode_ids[i]
                    cursor.execute(f"UPDATE mode SET parent_mode_id = {int(parents[j])} WHERE mode_id = {int(mode_ids[j])}")

    conn.commit()
    
                    
#bug do we need to close after ALL commits

In [ ]:
print(star)